In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
current_path = os.path.dirname(os.path.realpath(__file__)) if '__file__' in locals() else os.getcwd()
ROOT_DIR = os.path.abspath(os.path.join(current_path, '..'))
DATA_DIR = os.path.join(ROOT_DIR, 'data', 'raw')

Считаем датасет

In [3]:
customers = pd.read_csv(os.path.join(DATA_DIR, 'olist_customers_dataset.csv'))
geolocation = pd.read_csv(os.path.join(DATA_DIR, 'olist_geolocation_dataset.csv'))
order_items = pd.read_csv(os.path.join(DATA_DIR, 'olist_order_items_dataset.csv'))
order_payments = pd.read_csv(os.path.join(DATA_DIR, 'olist_order_payments_dataset.csv'))
orders = pd.read_csv(os.path.join(DATA_DIR, 'olist_orders_dataset.csv'))
products = pd.read_csv(os.path.join(DATA_DIR, 'olist_products_dataset.csv'))
sellers = pd.read_csv(os.path.join(DATA_DIR, 'olist_sellers_dataset.csv'))
category_name_t = pd.read_csv(os.path.join(DATA_DIR, 'product_category_name_translation.csv'))
review = pd.read_csv(os.path.join(DATA_DIR, 'olist_order_reviews_dataset.csv'))

Смотрим размеры таблиц

In [4]:
print('Размеры таблиц:')
print('Клиенты:',customers.shape)
print('Геолокация:',geolocation.shape)
print('Позиции заказов:',order_items.shape)
print('Платежи:',order_payments.shape)
print('Заказы:',orders.shape)
print('Товары:',products.shape)
print('Продавцы:',sellers.shape)
print('Перевод категорий:',category_name_t.shape)
print('Отзывы:',review.shape)

Размеры таблиц:
Клиенты: (99441, 5)
Геолокация: (1000163, 5)
Позиции заказов: (112650, 7)
Платежи: (103886, 5)
Заказы: (99441, 8)
Товары: (32951, 9)
Продавцы: (3095, 4)
Перевод категорий: (71, 2)
Отзывы: (99224, 7)


Смотрим на содержание таблиц

In [5]:
print('Колонки таблиц:')
print('Клиенты:',customers.columns)
print('Геолокация:',geolocation.columns)
print('Позиции заказов:',order_items.columns)
print('Платежи:',order_payments.columns)
print('Заказы:',orders.columns)
print('Товары:',products.columns)
print('Продавцы:',sellers.columns)
print('Перевод категорий:',category_name_t.columns)
print('Отзывы:',review.columns)

Колонки таблиц:
Клиенты: Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='str')
Геолокация: Index(['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
       'geolocation_city', 'geolocation_state'],
      dtype='str')
Позиции заказов: Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='str')
Платежи: Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='str')
Заказы: Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')
Товары: Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',


Посмотрим количество пропусков в таблицах

In [6]:
print('Количество null:')
print('customers:',customers.isnull().sum().sum())
print('geolocation:',geolocation.isnull().sum().sum())
print('order_items:',order_items.isnull().sum().sum())
print('order_payments:',order_payments.isnull().sum().sum())
print('orders:',orders.isnull().sum().sum())
print('products:',products.isnull().sum().sum())
print('sellers:',sellers.isnull().sum().sum())
print('category_name_t:',category_name_t.isnull().sum().sum())
print('review:',review.isnull().sum().sum())

Количество null:
customers: 0
geolocation: 0
order_items: 0
order_payments: 0
orders: 4908
products: 2448
sellers: 0
category_name_t: 0
review: 145903


Видим наличие пропусков в таблицах товаров, заказов и отзывов. И если наличие отзывов объясняется тем, что не каждый пользователь оставляет отзыв на купленный товар. Проверим таблицы orders и products.

In [7]:
print('orders:',orders.isnull().sum())
print('products:',products.isnull().sum())

orders: order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
products: product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


| Поле                         | Пропуски |
|------------------------------|----------|
| order_approved_at        | 160      |
| order_delivered_carrier_date          | 1783     |
| order_delivered_customer_date   | 2965     |

Столбец order_approved_at может быть null в случае отмены заказа (order_status = canceled)

Столбец order_delivered_carrier_date может быть null в случае если заказ еще не передан в доставку 

Столбец order_delivered_customer_date может быть null в случае если заказ еще не доехал до клиента 

Вместо заполнения или удаления пропусков создадим новые признаки

In [8]:
#Признак: заказ был доставлен?
orders['is_delivered'] = orders['order_delivered_customer_date'].notna().astype(int)

#Признак: заказ был одобрен?
orders['is_approved'] = orders['order_approved_at'].notna().astype(int)

#Признак: время доставки в днях (только для доставленных)
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])

orders['delivery_days'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.days

#Для недоставленных заказов будет -1
orders['delivery_days'] = orders['delivery_days'].fillna(-1)

#Признак: задержка доставки (факт > плана)
orders['is_delayed'] = (
    (orders['order_delivered_customer_date'] > orders['order_estimated_delivery_date']) & 
    (orders['is_delivered'] == 1)
).astype(int)

| Поле                         | Пропуски |
|------------------------------|----------|
| product_category_name        | 610      |
| product_name_lenght          | 610      |
| product_description_lenght   | 610      |
| product_photos_qty           | 610      |
| product_weight_g             | 2        |
| product_length_cm            | 2        |
| product_height_cm            | 2        |
| product_width_cm             | 2        |

In [9]:
#Пропуски по 610 заполним нулями
products['product_category_name'] = products['product_category_name'].fillna('unknown')
products['product_name_lenght'] = products['product_name_lenght'].fillna(0)
products['product_description_lenght'] = products['product_description_lenght'].fillna(0)
products['product_photos_qty'] = products['product_photos_qty'].fillna(0)

#Заполним медианой пустые колонки размеров
for col in ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']:
    median_val = products[col].median()
    products[col] = products[col].fillna(median_val)

Проверим снова пропуски

In [10]:
print('Количество null:')
print('customers:',customers.isnull().sum().sum())
print('geolocation:',geolocation.isnull().sum().sum())
print('order_items:',order_items.isnull().sum().sum())
print('order_payments:',order_payments.isnull().sum().sum())
print('orders:',orders.isnull().sum().sum())
print('products:',products.isnull().sum().sum())
print('sellers:',sellers.isnull().sum().sum())
print('category_name_t:',category_name_t.isnull().sum().sum())
print('review:',review.isnull().sum().sum())

Количество null:
customers: 0
geolocation: 0
order_items: 0
order_payments: 0
orders: 4908
products: 0
sellers: 0
category_name_t: 0
review: 145903


Видим, что пропуски остались только в отзывах

In [11]:
cust_orders = customers.merge(orders, on = 'customer_id', how = 'left')
cust_geo = customers.merge(geolocation, left_on = 'customer_zip_code_prefix', right_on = 'geolocation_zip_code_prefix', how = 'left')

Проверим данные на наличие дубликатов

In [14]:
print('Количество дубликатов:')
print('customers:',customers.duplicated().sum())
print('geolocation:',geolocation.duplicated().sum())
print('order_items:',order_items.duplicated().sum())
print('order_payments:',order_payments.duplicated().sum())
print('orders:',orders.duplicated().sum())
print('products:',products.duplicated().sum())
print('sellers:',sellers.duplicated().sum())
print('category_name_t:',category_name_t.duplicated().sum())
print('review:',review.duplicated().sum())

Количество дубликатов:
customers: 0
geolocation: 261831
order_items: 0
order_payments: 0
orders: 0
products: 0
sellers: 0
category_name_t: 0
review: 0


Видим, что дубликатов нет нигде кроме геолокации, это нормально так как многие заказы могли приходить примерно на одно место

Далее проверим на наличие выбросов. Сделаем это при помощи визуализации Boxplot графика

Перед проверкой соберем главный датасет

In [18]:
#Преобразуем строки в даты 
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])

#Создаем признаки, которые были в EDA
orders['is_delivered'] = orders['order_delivered_customer_date'].notna().astype(int)
orders['is_approved'] = orders['order_approved_at'].notna().astype(int)

#Время доставки в днях
orders['delivery_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days
orders['delivery_days'] = orders['delivery_days'].fillna(-1)

#Задержка доставки
orders['is_delayed'] = (
    (orders['order_delivered_customer_date'] > orders['order_estimated_delivery_date']) & 
    (orders['is_delivered'] == 1)
).astype(int)

#Для числовых признаков
numeric_cols = ['price', 'freight_value', 'payment_value', 'product_weight_g']

In [16]:
payments_agg = order_payments.groupby('order_id').agg({
    'payment_value': 'sum',
    'payment_installments': 'max',
    'payment_sequential': 'max'
}).reset_index()

# Собираем главный датасет
df_model = (orders
            .merge(order_items, on='order_id', how='inner')
            .merge(products, on='product_id', how='left')
            .merge(payments_agg, on='order_id', how='left')
            .merge(customers, on='customer_id', how='left')
            .merge(sellers, on='seller_id', how='left')
)

Построение boxplot

In [21]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(['price', 'freight_value', 'payment_value', 'product_weight_g']):
    # Boxplot
    sns.boxplot(data=df_model, y=col, ax=axes[i], color='skyblue')
    axes[i].set_title(f'Boxplot: {col}')
    axes[i].set_ylabel('Значение')

plt.tight_layout()
plt.show()

KeyboardInterrupt: 

Из графика видно, что есть выбросы, но в условиях маркетплейса считаю их нормальными, так как бОльшее число товаров дешевые, а выбросы являются просто дорогими товарами, которые люди редко заказывают. Я знаю 2 метода для поиска выбросов - IQR и Z-score. В нашем слчуае (не нормальное, а скошенное распределие) больше подходит IQR, но покажу результаты двух вариантов.

IQR

In [22]:
def detect_outliers_iqr_fixed(df, column, k=3.0):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    
    # Стандартная формула
    lower_bound_raw = Q1 - k * IQR
    upper_bound = Q3 + k * IQR
    
    # 🟢 ИСПРАВЛЕНИЕ: Если граница ушла в минус, ставим 0
    lower_bound = max(0, lower_bound_raw)
    
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    
    return {
        'count': len(outliers),
        'percent': len(outliers) / len(df) * 100,
        'bounds': (lower_bound, upper_bound)
    }

print("🔍 Выбросы по методу IQR (с учетом физики данных):")
print(f"{'Признак':<20} {'Выбросов':>10} {'% от выборки':>15} {'Диапазон нормы':>25}")
print("-" * 75)

for col in ['price', 'freight_value', 'payment_value', 'product_weight_g']:
    result = detect_outliers_iqr_fixed(df_model, col, k=3.0)
    bounds = result['bounds']
    print(f"{col:<20} {result['count']:>10,} {result['percent']:>14.1f}% [{bounds[0]:.1f}; {bounds[1]:.1f}]")

🔍 Выбросы по методу IQR (с учетом физики данных):
Признак                Выбросов    % от выборки            Диапазон нормы
---------------------------------------------------------------------------
price                     4,074            3.6% [0.0; 419.9]
freight_value             5,538            4.9% [0.0; 45.4]
payment_value             4,911            4.4% [0.0; 584.5]
product_weight_g         11,106            9.9% [0.0; 6300.0]


Z-score

In [20]:
from scipy import stats

print("\nВыбросы по методу Z-score (|Z| > 3):")
for col in numeric_cols:
    z_scores = stats.zscore(df_model[col].dropna())
    outliers = np.abs(z_scores) > 3
    print(f"{col:<20}: {outliers.sum():,} выбросов ({outliers.sum()/len(df_model)*100:.1f}%)")


🔍 Выбросы по методу Z-score (|Z| > 3):
price               : 1,966 выбросов (1.7%)
freight_value       : 2,041 выбросов (1.8%)
payment_value       : 1,786 выбросов (1.6%)
product_weight_g    : 2,955 выбросов (2.6%)


Видим, что результаты сильно отличаются, более точные у IQR и выбросы (если смотреть на суть датасета) некритичные, поэтому мы не будем удалять данные. Если мы их удалим, то модель не научится работать с дорогими товарами.

Построим график распределения клиентов по штатам

In [13]:
state_counts = customers['customer_state'].value_counts()

plt.figure(figsize=(10, 6))
sns.barplot(x=state_counts.index, y=state_counts.values)
plt.title('Распределение клиентов по штатам')
plt.xlabel('Штат')
plt.ylabel('Количество клиентов')
plt.xticks(rotation=45)
plt.show()
#plt.savefig('hseml-group-project-erste-ritter-team/data/eda_plots/barplot-stats.png')

FileNotFoundError: [Errno 2] No such file or directory: 'hseml-group-project-erste-ritter-team/data/eda_plots/barplot-stats.png'

Построим график топ-10 городов по количеству клиентов

In [ ]:
top_cities = customers['customer_city'].value_counts().head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_cities.values, y=top_cities.index, palette='magma')
plt.title('Топ-10 городов по количеству клиентов')
plt.xlabel('Количество клиентов')
plt.ylabel('Город')
plt.show()
#plt.savefig('hseml-group-project-erste-ritter-team/data/eda_plots/top10-cities.png')

Построим корреляционную матрицу платежей

In [ ]:
orders_payments = orders.merge(order_payments, on='order_id')
numeric_data = orders_payments.select_dtypes(include=['float64', 'int64'])
corr_matrix = numeric_data.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, linewidths=.5, fmt='.2f')
plt.title('Корреляционная матрица')
plt.show()
#plt.savefig('hseml-group-project-erste-ritter-team/data/eda_plots/corr_matrix.png')